# This notebook describe the basic information of the project dataset..

In [1]:
lazy import pandas as pd


## 1. Dataset Overview

### Objective
Understand the structure, size, and data types of the dataset.

### Why?
This helps identify the available features and determines the appropirate preprocessing techniques.

In [2]:
df = pd.read_csv('../data/raw/Student_Performance.csv')
print("Dataset loaded successfulyy!")

Dataset loaded successfulyy!


In [3]:
df.sample(6)


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
548,5,72,No,9,9,60.0
5165,6,52,No,4,4,35.0
7129,8,42,Yes,7,4,37.0
7151,1,95,No,8,3,71.0
3884,6,95,Yes,6,7,87.0
2992,2,95,No,7,7,71.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Hours Studied                     10000 non-null  int64  
 1   Previous Scores                   10000 non-null  int64  
 2   Extracurricular Activities        10000 non-null  str    
 3   Sleep Hours                       10000 non-null  int64  
 4   Sample Question Papers Practiced  10000 non-null  int64  
 5   Performance Index                 10000 non-null  float64
dtypes: float64(1), int64(4), str(1)
memory usage: 468.9 KB


In [5]:
columns = df.columns.to_list
columns

<bound method IndexOpsMixin.tolist of Index(['Hours Studied', 'Previous Scores', 'Extracurricular Activities',
       'Sleep Hours', 'Sample Question Papers Practiced', 'Performance Index'],
      dtype='str')>

### Observation 
- The dataset contains 10,000 rows and 6 columns.
- No Missing Values were detected.

## 2. Statictical information of dataset.

In [6]:
df.describe()

,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Performance Index
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,4.992900,69.445700,6.530600,4.583300,55.224800
std,2.589309,17.343152,1.695863,2.867348,19.212558
min,1.000000,40.000000,4.000000,0.000000,10.000000
25%,3.000000,54.000000,5.000000,2.000000,40.000000
50%,5.000000,69.000000,7.000000,5.000000,55.000000
75%,7.000000,85.000000,8.000000,7.000000,71.000000
max,9.000000,99.000000,9.000000,9.000000,100.000000


## 3. Data Quality Analysis

### 3.1 Duplicate Records

Duplicate records are rows that contain identical values
across all selected columns.

### Objective
Identify and investigate duplicate rows before deciding
whether any records should be removed.

In [7]:
duplicate_rows = df[df.duplicated(keep=False)]

duplicate_rows.shape

(253, 6)

In [8]:
df[df.duplicated(keep=False)].sort_values(
    by=df.columns.tolist()
).head(10)

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
6975,1,41,No,7,3,12.0
9966,1,41,No,7,3,12.0
2225,1,42,Yes,9,7,17.0
4986,1,42,Yes,9,7,17.0
2647,1,57,No,8,0,31.0
8522,1,57,No,8,0,31.0
3398,1,63,Yes,8,3,37.0
8184,1,63,Yes,8,3,37.0
4573,1,66,No,8,2,40.0
5578,1,66,No,8,2,40.0


In [9]:
df.drop_duplicates(inplace=True)
df.shape

C:\Users\shekh\AppData\Local\Temp\ipykernel_23044\3054484884.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop_duplicates(inplace=True)


(9873, 6)

### 3.2 Missing Value Analysis

Missing values occur when a feature has no recorded value.

### Objective
Determine whether any columns require missing-value treatment.

In [10]:
df.isnull().sum()

Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64

### Observation
- No missing values were detected using isnull().
- No missing-value imputation is required based on this check.
- Invalid values and inconsistent categories still need investigation.


In [11]:
df.to_csv('df.csv',index=False)

In [13]:
import pandas as pd
import numpy as np



# Convert Yes/No into 1/0
extra = df["Extracurricular Activities"].map({
    "Yes": 1,
    "No": 0
})

# Published approximate regression equation
df["Calculated_PI"] = (
    -34.0756
    + 2.8530 * df["Hours Studied"]
    + 1.0184 * df["Previous Scores"]
    + 0.4806 * df["Sleep Hours"]
    + 0.1938 * df["Sample Question Papers Practiced"]
    + 0.6129 * extra
)

# Compare calculated values with original target
df["Difference"] = (
    df["Performance Index"] - df["Calculated_PI"]
).abs()

print("Mean Absolute Difference:",
      df["Difference"].mean())

print("Maximum Difference:",
      df["Difference"].max())

display(df[[
    "Performance Index",
    "Calculated_PI",
    "Difference"
]].head(10))

Mean Absolute Difference: 1.6231385900941964
Maximum Difference: 8.794400000000003


,Performance Index,Calculated_PI,Difference
0,91.0,91.8491,0.8491
1,65.0,63.1552,1.8448
2,45.0,45.0515,0.0515
3,36.0,36.5497,0.5497
4,66.0,67.0892,1.0892
5,61.0,59.4068,1.5932
6,63.0,64.4173,1.4173
7,42.0,38.2745,3.7255
8,61.0,62.8386,1.8386
9,69.0,69.8964,0.8964


In [15]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error



X = df.drop(columns=["Performance Index"]).copy()
y = df["Performance Index"]

# Convert categorical feature into numeric
X["Extracurricular Activities"] = (
    X["Extracurricular Activities"].map({"Yes": 1, "No": 0})
)

# Fit Linear Regression on the full dataset for this diagnostic
model = LinearRegression()
model.fit(X, y)

predictions = model.predict(X)

print("Intercept:", model.intercept_)

print("\nCoefficients:")
for feature, coefficient in zip(X.columns, model.coef_):
    print(f"{feature}: {coefficient:.6f}")

print("\nR²:", r2_score(y, predictions))
print("MAE:", mean_absolute_error(y, predictions))

print("\nFirst 10 comparisons:")
comparison = pd.DataFrame({
    "Actual_PI": y,
    "Predicted_PI": predictions,
})
comparison["Absolute_Difference"] = (
    comparison["Actual_PI"] - comparison["Predicted_PI"]
).abs()

display(comparison.head(10))

Intercept: -3.155174553099016

Coefficients:
Hours Studied: 0.263020
Previous Scores: 0.093880
Extracurricular Activities: 0.060545
Sleep Hours: 0.043976
Sample Question Papers Practiced: 0.018038
Calculated_PI: 0.907743
Difference: 0.010774

R²: 0.9886818660796503
MAE: 1.6230120769057206

First 10 comparisons:


,Actual_PI,Predicted_PI,Absolute_Difference
0,91.0,91.839063,0.839063
1,65.0,63.155674,1.844326
2,45.0,45.037101,0.037101
3,36.0,36.541882,0.541882
4,66.0,67.080512,1.080512
5,61.0,59.403868,1.596132
6,63.0,64.417541,1.417541
7,42.0,38.301846,3.698154
8,61.0,62.837739,1.837739
9,69.0,69.885823,0.885823
